<a href="https://colab.research.google.com/github/20punhirune/MLlearning-notebooks/blob/main/lecture03_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第3回講義 宿題

## 課題

今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) を多層パーセプトロンによって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値

Accuracy 85%

### ルール

- 訓練データは`x_train`， `t_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train`，`t_train`以外の学習データは使わないでください．**
- **多層パーセプトロンのアルゴリズム部分は第3回の演習を参考に，NumPyのみで実装してください．** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルをcsv形式で保存し，**Omnicampusの宿題タブから「第3回 ニューラルネットワーク基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを　ファイル＞ダウンロード＞.pyをダウンロード　から保存し，**Omnicampusの宿題タブから「第3回 ニューラルネットワーク基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコード全体をコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

### 評価方法
- 予測ラベルの`t_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [49]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
# 作業ディレクトリを指定
work_dir = 'drive/MyDrive/Colab Notebooks/DLBasics2026_colab'

### データの読み込み（このセルは修正しないでください）

In [51]:
import os
import numpy as np
import pandas as pd
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import inspect


#学習データ
x_train = np.load(work_dir + '/Lecture03/data/x_train.npy')
t_train = np.load(work_dir + '/Lecture03/data/y_train.npy')

#テストデータ
x_test = np.load(work_dir + '/Lecture03/data/x_test.npy')

# データの前処理（正規化， one-hot encoding)
x_train, x_test = x_train / 255., x_test / 255.
x_train, x_test = x_train.reshape(x_train.shape[0], -1), x_test.reshape(x_test.shape[0], -1)
t_train = np.eye(N=10)[t_train.astype("int32").flatten()]

### 多層パーセプトロンの実装

In [52]:
# データの分割
x_train, x_val, t_train, t_val =\
    train_test_split(x_train, t_train, test_size=10000, random_state=42)

In [53]:
def np_log(x):
    return np.log(np.clip(x, 1e-10, 1e+10))


def create_batch(data, batch_size):
    """
    :param data: np.ndarray，入力データ
    :param batch_size: int，バッチサイズ
    """
    num_batches, mod = divmod(data.shape[0], batch_size)
    batched_data = np.split(data[: batch_size * num_batches], num_batches)
    if mod:
        batched_data.append(data[batch_size * num_batches:])

    return batched_data

In [54]:
# シード値を変えることで何が起きるかも確かめてみてください．
rng = np.random.RandomState(1234)
random_state = 42


# 発展: 今回の講義で扱っていない活性化関数について調べ，実装してみましょう
def relu(x):
    # WRITE ME
    return np.maximum(x, 0)


def deriv_relu(x):
    # WRITE ME
    return (x>0).astype(x.dtype)


def softmax(x):
    # WRITE ME
    x -= x.max(axis=1, keepdims=True)
    x_exp = np.exp(x)
    return x_exp / np.sum(x_exp, axis=1, keepdims=True)


def deriv_softmax(x):
    # WRITE ME
    return softmax(x) * (1-softmax(x))


def crossentropy_loss(t, y):
    # WRITE ME
    return  (- t * np_log(y)).sum(axis=1).mean()


class Dense:
    # WRITE ME
    # 何を使うかを受け取って登録している
    def __init__(self, in_dim, out_dim, function, deriv_function):
        # He initializationを適用
        self.W = rng.normal(0.0, np.sqrt(2.0 / in_dim), size=(in_dim, out_dim))
        self.b = np.zeros(out_dim).astype("float64")
        self.function = function
        self.deriv_function = deriv_function

        self.x = None
        self.u = None

        self.dW = None
        self.db = None

        self.params_idxs = np.cumsum([self.W.size, self.b.size])

        #### Adam ####
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)

        # 更新回数のカウンタ
        self.t = 0
        ##############


    def __call__(self, x):
        self.x = x
        self.u = np.matmul(self.x, self.W) + self.b
        h = self.function(self.u)
        return h

    def b_prop(self, delta, W):
        self.delta = self.deriv_function(self.u) * np.matmul(delta, W.T) # WRITE ME
        return self.delta

    def compute_grad(self):
        batch_size = self.delta.shape[0]

        self.dW = np.matmul(self.x.T, self.delta) / batch_size # WRITE ME
        self.db = np.matmul(np.ones(batch_size), self.delta) / batch_size # WRITE ME

    def get_params(self):
        return np.concatenate([self.W.ravel(), self.b], axis=0)

    def set_params(self, params):
        _W, _b = np.split(params, self.params_idxs)[:-1]
        self.W = _W.reshape(self.W.shape)
        self.b = _b

    def get_grads(self):
        return np.concatenate([self.dW.ravel(), self.db], axis=0)

class Model:
    def __init__(self, hidden_dims, activation_functions, deriv_functions):
        # 各層をリストに格納していく
        self.layers = []
        for i in range(len(hidden_dims)-2):  # 出力層以外は同じ構造
            self.layers.append(Dense(hidden_dims[i], hidden_dims[i+1],
                                     activation_functions[i], deriv_functions[i]))
        # 最後の出力層だけfor文の外で追加する
        self.layers.append(Dense(hidden_dims[-2], hidden_dims[-1],
                                 activation_functions[-1], deriv_functions[-1]))

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        # 順伝播処理を行うメソッド
        for layer in self.layers:
            x = layer(x)
        return x

    def backward(self, delta):
        # 誤差逆伝播，勾配計算を行うメソッド
        batch_size = delta.shape[0]

        for i, layer in enumerate(self.layers[::-1]):
            if i == 0:  # 出力層の場合
                layer.delta = delta  # y - t
                layer.compute_grad()
            else:  # 出力層以外の場合
                delta = layer.b_prop(delta, W)  # 逆伝播
                layer.compute_grad()  # 勾配の計算

            W = layer.W

    def update(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        # パラメータの更新を行うメソッド
        for layer in self.layers:
            layer.t += 1

            #### Adam ####
            layer.mW = beta1 * layer.mW + (1-beta1) * layer.dW
            layer.vW = beta2 * layer.vW + (1-beta2) * layer.dW**2
            mW_hat = layer.mW / (1 - beta1**(layer.t))
            vW_hat = layer.vW / (1 - beta2**(layer.t))

            layer.W -=  lr * mW_hat / (np.sqrt(vW_hat) + eps)

            layer.mb = beta1 * layer.mb + (1-beta1) * layer.db
            layer.vb = beta2 * layer.vb + (1-beta2) * layer.db**2
            mb_hat = layer.mb / (1 - beta1**(layer.t))
            vb_hat = layer.vb / (1 - beta2**(layer.t))

            layer.b -= lr * mb_hat / (np.sqrt(vb_hat) + eps)
            ##############

            # layer.W -= lr * layer.dW
            # layer.b -= lr * layer.db

# lr = 0.1 # SGD用
lr = 1e-3 # Adam用
n_epochs = 30 # WRITE ME
batch_size = 128 # WRITE ME

# WRITE ME
mlp = Model(
    hidden_dims = [784, 100, 100, 10],
    activation_functions = [relu, relu, softmax],
    deriv_functions = [deriv_relu, deriv_relu, deriv_softmax]
)

In [55]:
# x = x_train[:128]
# t = t_train[:128]

# y = mlp(x)
# mlp.backward(y - t)

# l = mlp.layers[0]
# print("t before:", l.t)          # 0 のはず
# W_before = l.W.copy()

# mlp.update(1e-3)

# print("t after:", l.t)           # 1 のはず
# mask = np.abs(l.dW) > 1e-4       # eps の影響をさらに小さくするため、しきい値を 1e-4 に
# ratio = np.abs(l.W - W_before)[mask] / 1e-3
# print(ratio.min(), ratio.max())  # どちらも 1.0 に近くなるはず

# b_before = l.b.copy()      # update の前にコピー（W_before の隣に置く）

# mlp.update(1e-3)

# mask_b = np.abs(l.db) > 1e-4
# ratio_b = np.abs(l.b - b_before)[mask_b] / 1e-3
# print(ratio_b.min(), ratio_b.max())   # どちらも 1.0 に近くなるはず

### モデルの学習

In [56]:
def train_model(model, x_train, t_train, x_val, t_val, n_epochs=10):
    for epoch in range(n_epochs):
        losses_train = []
        losses_valid = []
        train_num = 0
        train_true_num = 0
        valid_num = 0
        valid_true_num = 0

        x_train, t_train = shuffle(x_train, t_train, random_state=rng)
        x_train_batches, t_train_batches = create_batch(x_train, batch_size), create_batch(t_train, batch_size)

        x_val, t_val = shuffle(x_val, t_val)
        x_val_batches, t_val_batches = create_batch(x_val, batch_size), create_batch(t_val, batch_size)

        # モデルの訓練
        for x, t in zip(x_train_batches, t_train_batches):
            # 順伝播
            # WRITE ME
            y = model(x)

            # 損失の計算
            # WRITE ME
            loss = crossentropy_loss(t,y)
            losses_train.append(loss.tolist())

            # 逆伝播
            # WRITE ME
            delta = y - t
            model.backward(delta)

            # パラメータの更新
            # WRITE ME
            model.update(lr)

            # 精度を計算
            acc = accuracy_score(t.argmax(axis=1), y.argmax(axis=1), normalize=False)
            train_num += x.shape[0]
            train_true_num += acc

        # モデルの評価
        for x, t in zip(x_val_batches, t_val_batches):
            # 順伝播
            # WRITE ME
            y = model(x)

            # 損失の計算
            # WRITE ME
            loss = crossentropy_loss(t,y)
            losses_valid.append(loss.tolist())

            acc = accuracy_score(t.argmax(axis=1), y.argmax(axis=1), normalize=False)
            valid_num += x.shape[0]
            valid_true_num += acc

        print('EPOCH: {}, Train [Loss: {:.3f}, Accuracy: {:.3f}], Valid [Loss: {:.3f}, Accuracy: {:.3f}]'.format(
            epoch,
            np.mean(losses_train),
            train_true_num/train_num,
            np.mean(losses_valid),
            valid_true_num/valid_num
        ))


train_model(mlp, x_train, t_train, x_val, t_val, n_epochs)

EPOCH: 0, Train [Loss: 0.584, Accuracy: 0.798], Valid [Loss: 0.425, Accuracy: 0.848]
EPOCH: 1, Train [Loss: 0.405, Accuracy: 0.857], Valid [Loss: 0.389, Accuracy: 0.863]
EPOCH: 2, Train [Loss: 0.365, Accuracy: 0.871], Valid [Loss: 0.353, Accuracy: 0.876]
EPOCH: 3, Train [Loss: 0.339, Accuracy: 0.877], Valid [Loss: 0.371, Accuracy: 0.866]
EPOCH: 4, Train [Loss: 0.313, Accuracy: 0.886], Valid [Loss: 0.345, Accuracy: 0.876]
EPOCH: 5, Train [Loss: 0.302, Accuracy: 0.890], Valid [Loss: 0.342, Accuracy: 0.872]
EPOCH: 6, Train [Loss: 0.289, Accuracy: 0.894], Valid [Loss: 0.327, Accuracy: 0.881]
EPOCH: 7, Train [Loss: 0.274, Accuracy: 0.899], Valid [Loss: 0.323, Accuracy: 0.881]
EPOCH: 8, Train [Loss: 0.264, Accuracy: 0.903], Valid [Loss: 0.331, Accuracy: 0.881]
EPOCH: 9, Train [Loss: 0.253, Accuracy: 0.907], Valid [Loss: 0.318, Accuracy: 0.888]
EPOCH: 10, Train [Loss: 0.247, Accuracy: 0.907], Valid [Loss: 0.311, Accuracy: 0.890]
EPOCH: 11, Train [Loss: 0.239, Accuracy: 0.910], Valid [Loss: 0.

In [57]:
t_pred = []
for x in x_test:
    # 順伝播
    x = x[np.newaxis, :]
    y = mlp(x)

    # モデルの出力を予測値のスカラーに変換
    pred = y.argmax(1).tolist()

    t_pred.extend(pred)

submission = pd.Series(t_pred, name='label')
submission.to_csv(work_dir + '/Lecture03/submission_pred_03.csv', header=True, index_label='id')